# Fase 4 - Análise Comparativa Multi-Modelo

Este notebook analisa os resultados dos experimentos da Fase 4, comparando diferentes modelos NLG (Gemma, Llama, Mistral, Phi) executando a mesma persona freiriana.

## Objetivos da Análise:
1. **Performance Técnica**: Latência, uso de VRAM/RAM
2. **Qualidade Pedagógica**: Manutenção de persona, scaffolding, uso de ferramentas
3. **Comparação com Baseline**: Gemma vs. outros modelos
4. **Identificação de Problemas**: Loops, quebra de regras, etc.

In [ ]:
# Imports
import pandas as pd
import json
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing import Dict, List, Any

# Configuração de visualização
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
%matplotlib inline

## 1. Carregar Dados dos Experimentos

In [ ]:
def load_experiment_data(base_log_dir: str = "../../../logs/experiments") -> Dict[str, pd.DataFrame]:
    """
    Carrega dados de todos os experimentos e retorna DataFrames por modelo.
    """
    base_path = Path(base_log_dir)
    all_data = {}
    
    # Procurar por diretórios de experimentos
    for experiment_dir in base_path.glob("*"):
        if experiment_dir.is_dir():
            # Cada subdiretório é um modelo
            for model_dir in experiment_dir.glob("*"):
                if model_dir.is_dir():
                    interactions_file = model_dir / "interactions.jsonl"
                    if interactions_file.exists():
                        # Carregar JSONL
                        interactions = []
                        with open(interactions_file, 'r', encoding='utf-8') as f:
                            for line in f:
                                if line.strip():
                                    interactions.append(json.loads(line))
                        
                        # Converter para DataFrame
                        df = pd.DataFrame(interactions)
                        model_name = model_dir.name
                        all_data[model_name] = df
                        print(f"✓ Carregado: {model_name} ({len(df)} interações)")
    
    return all_data

# Carregar dados
data = load_experiment_data()
print(f"\nTotal de modelos: {len(data)}")

## 2. Performance Técnica - Tabela Comparativa

In [ ]:
def compute_technical_metrics(data: Dict[str, pd.DataFrame]) -> pd.DataFrame:
    """
    Calcula métricas técnicas por modelo.
    """
    metrics = []
    
    for model_name, df in data.items():
        metrics.append({
            "Modelo": model_name,
            "Latência Média (ms)": df['latency_ms'].mean(),
            "Latência Min (ms)": df['latency_ms'].min(),
            "Latência Max (ms)": df['latency_ms'].max(),
            "VRAM Máxima (MB)": df['vram_used_mb'].max() if 'vram_used_mb' in df.columns else 0,
            "VRAM Média (MB)": df['vram_used_mb'].mean() if 'vram_used_mb' in df.columns else 0,
            "RAM Máxima (MB)": df['ram_used_mb'].max() if 'ram_used_mb' in df.columns else 0,
            "Total de Turnos": len(df),
        })
    
    return pd.DataFrame(metrics).round(2)

# Gerar tabela
technical_metrics = compute_technical_metrics(data)
print("\n=== TABELA DE PERFORMANCE TÉCNICA ===")
display(technical_metrics)

## 3. Visualização - Latência Comparativa

In [ ]:
# Gráfico de barras - Latência média
plt.figure(figsize=(12, 6))
sns.barplot(data=technical_metrics, x='Modelo', y='Latência Média (ms)', palette='viridis')
plt.title('Latência Média por Modelo (ms)', fontsize=16, fontweight='bold')
plt.xlabel('Modelo NLG', fontsize=12)
plt.ylabel('Latência Média (ms)', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Box plot - Distribuição de latências
plt.figure(figsize=(14, 6))
latency_data = pd.concat([df[['model_name', 'latency_ms']] for df in data.values()])
sns.boxplot(data=latency_data, x='model_name', y='latency_ms', palette='Set2')
plt.title('Distribuição de Latência por Modelo', fontsize=16, fontweight='bold')
plt.xlabel('Modelo NLG', fontsize=12)
plt.ylabel('Latência (ms)', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 4. Visualização - Uso de Memória (VRAM)

In [ ]:
# Gráfico de barras - VRAM máxima
plt.figure(figsize=(12, 6))
sns.barplot(data=technical_metrics, x='Modelo', y='VRAM Máxima (MB)', palette='rocket')
plt.title('Uso Máximo de VRAM por Modelo (MB)', fontsize=16, fontweight='bold')
plt.xlabel('Modelo NLG', fontsize=12)
plt.ylabel('VRAM Máxima (MB)', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 5. Análise Qualitativa - Distribuição de Traces

Os traces indicam qual rota o agente tomou:
- `EXECUTED_STANDARD`: Rota padrão (baseada na intenção NLU)
- `EXECUTED_SCAFFOLDING`: Rota de suporte adicional
- `EXECUTED_REACT_TOOL`: Uso de ferramenta externa (busca)
- `REACT_LOOP`: **PROBLEMA** - Modelo entrou em loop (como Gemma)

In [ ]:
def analyze_traces(data: Dict[str, pd.DataFrame]) -> pd.DataFrame:
    """
    Analisa a distribuição de traces (rotas) por modelo.
    """
    trace_data = []
    
    for model_name, df in data.items():
        trace_counts = df['agent_trace'].value_counts().to_dict()
        total = len(df)
        
        for trace, count in trace_counts.items():
            trace_data.append({
                "Modelo": model_name,
                "Trace": trace,
                "Contagem": count,
                "Percentual (%)": round((count / total) * 100, 1)
            })
    
    return pd.DataFrame(trace_data)

# Gerar tabela de traces
trace_analysis = analyze_traces(data)
print("\n=== DISTRIBUIÇÃO DE TRACES (ROTAS DO AGENTE) ===")
display(trace_analysis)

In [ ]:
# Visualização - Distribuição de traces
plt.figure(figsize=(14, 6))
sns.barplot(data=trace_analysis, x='Modelo', y='Percentual (%)', hue='Trace', palette='pastel')
plt.title('Distribuição de Rotas (Traces) por Modelo', fontsize=16, fontweight='bold')
plt.xlabel('Modelo NLG', fontsize=12)
plt.ylabel('Percentual (%)', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.legend(title='Trace', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 6. Análise Qualitativa - Inspeção Manual de Diálogos

Para análise qualitativa detalhada, vamos imprimir alguns diálogos lado a lado.

In [ ]:
def print_dialogue_comparison(data: Dict[str, pd.DataFrame], turn_number: int = 1):
    """
    Imprime lado a lado as respostas de todos os modelos para um mesmo turno.
    """
    print(f"\n{'='*100}")
    print(f"COMPARAÇÃO - TURNO {turn_number}")
    print(f"{'='*100}\n")
    
    for model_name, df in data.items():
        if turn_number <= len(df):
            row = df.iloc[turn_number - 1]
            print(f"\n--- {model_name} ---")
            print(f"[ALUNO]: {row['user_input']}")
            print(f"[LEIA]:  {row['agent_response']}")
            print(f"         (Trace: {row['agent_trace']}, Latência: {row['latency_ms']}ms)")
            print("-" * 100)

# Comparar primeiro turno
print_dialogue_comparison(data, turn_number=1)

# Comparar outros turnos importantes
# print_dialogue_comparison(data, turn_number=3)
# print_dialogue_comparison(data, turn_number=5)

## 7. Análise de Confiança NLU

In [ ]:
# Distribuição de confiança do classificador NLU
plt.figure(figsize=(14, 6))
nlu_data = pd.concat([df[['model_name', 'nlu_confidence']] for df in data.values()])
sns.boxplot(data=nlu_data, x='model_name', y='nlu_confidence', palette='coolwarm')
plt.title('Distribuição de Confiança do Classificador NLU', fontsize=16, fontweight='bold')
plt.xlabel('Modelo NLG', fontsize=12)
plt.ylabel('Confiança NLU', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Nota: A confiança NLU deve ser similar entre modelos, pois o classificador é o mesmo

## 8. Exportar Resultados para o TCC

Salvar tabelas e gráficos para usar no documento final.

In [ ]:
# Criar diretório para resultados
output_dir = Path("results")
output_dir.mkdir(exist_ok=True)

# Salvar tabelas em CSV
technical_metrics.to_csv(output_dir / "technical_metrics.csv", index=False)
trace_analysis.to_csv(output_dir / "trace_analysis.csv", index=False)

print("✓ Resultados exportados para:", output_dir.absolute())

## 9. Checklist de Avaliação Qualitativa (Manual)

Para cada modelo, revisar manualmente os diálogos e preencher esta tabela:

| Critério | Gemma | Llama | Mistral | Phi |
|----------|-------|-------|---------|-----|
| **Manutenção de Persona** (sempre faz perguntas) | [ ] | [ ] | [ ] | [ ] |
| **Nunca dá resposta direta** | [ ] | [ ] | [ ] | [ ] |
| **Scaffolding funciona** (quando aluno diz "não sei") | [ ] | [ ] | [ ] | [ ] |
| **Sem loops** (ReAct não trava) | [ ] | [ ] | [ ] | [ ] |
| **Tom acolhedor** | [ ] | [ ] | [ ] | [ ] |
| **Valida conhecimento prévio do aluno** | [ ] | [ ] | [ ] | [ ] |

**Nota**: Preencher após revisar os diálogos impressos acima.

## 10. Conclusões Preliminares

### Performance Técnica
- **Modelo mais rápido**: [Preencher após análise]
- **Modelo com menor uso de VRAM**: [Preencher após análise]
- **Trade-off ideal**: [Preencher após análise]

### Qualidade Pedagógica
- **Melhor manutenção de persona**: [Preencher após análise]
- **Melhor scaffolding**: [Preencher após análise]
- **Problemas identificados**: [Preencher após análise]

### Recomendação Final
[Preencher após análise completa]